# Taller resuelto: Mean Shift

## Objetivo
Comprender Mean Shift como algoritmo basado en densidad, aplicar bandwidth y comparar resultados.

## Bloque 1. Importar librerías

Cargaremos herramientas para generar datos, estimar bandwidth, entrenar Mean Shift y graficar.

In [ ]:
# Importamos NumPy.
import numpy as np

# Importamos Matplotlib.
import matplotlib.pyplot as plt

# Importamos datos sintéticos.
from sklearn.datasets import make_blobs

# Importamos MeanShift y estimate_bandwidth.
from sklearn.cluster import MeanShift, estimate_bandwidth

# Importamos Silhouette.
from sklearn.metrics import silhouette_score


### Interpretación

Mean Shift no necesita definir K, pero depende del bandwidth, que controla el tamaño de la ventana de densidad.

## Bloque 2. Generar datos sintéticos

Crearemos grupos compactos para observar cómo Mean Shift detecta regiones densas.

In [ ]:
# Generamos datos sintéticos.
X, y_true = make_blobs(n_samples=500, centers=4, cluster_std=0.8, random_state=42)

# Mostramos dimensiones.
print("Shape de X:", X.shape)

# Creamos figura.
plt.figure(figsize=(7,5))

# Dibujamos datos con grupos reales como referencia.
plt.scatter(X[:,0], X[:,1], c=y_true, cmap="viridis", edgecolors="k")

# Agregamos título.
plt.title("Datos sintéticos de referencia")

# Mostramos gráfica.
plt.show()


### Interpretación

Los datos tienen zonas de alta concentración. Mean Shift buscará esos máximos locales de densidad.

## Bloque 3. Estimar bandwidth

Calcularemos un bandwidth inicial con `estimate_bandwidth`.

In [ ]:
# Estimamos bandwidth con una muestra de los datos.
bandwidth = estimate_bandwidth(X, quantile=0.2, n_samples=500)

# Mostramos bandwidth.
print("Bandwidth estimado:", bandwidth)


### Interpretación

Bandwidth pequeño tiende a generar más clusters. Bandwidth grande tiende a fusionarlos.

## Bloque 4. Entrenar Mean Shift

Entrenaremos Mean Shift con el bandwidth estimado.

In [ ]:
# Creamos el modelo.
model = MeanShift(bandwidth=bandwidth, bin_seeding=True)

# Entrenamos y obtenemos etiquetas.
labels = model.fit_predict(X)

# Guardamos centros.
centers = model.cluster_centers_

# Mostramos clusters encontrados.
print("Clusters encontrados:", np.unique(labels))

# Mostramos cantidad de clusters.
print("Número de clusters:", len(centers))


### Interpretación

Mean Shift encontró automáticamente el número de clusters según la densidad observada.

## Bloque 5. Visualizar resultado

Graficaremos los clusters y centros encontrados.

In [ ]:
# Creamos figura.
plt.figure(figsize=(7,5))

# Dibujamos puntos según cluster.
plt.scatter(X[:,0], X[:,1], c=labels, cmap="viridis", edgecolors="k")

# Dibujamos centros.
plt.scatter(centers[:,0], centers[:,1], c="red", s=250, marker="X", label="Centros")

# Agregamos título y leyenda.
plt.title("Mean Shift - Clusters encontrados")
plt.legend()

# Mostramos gráfica.
plt.show()


### Interpretación

Los centros se ubican en regiones de alta densidad. El modelo no requiere K, pero sí una escala adecuada de densidad.

## Bloque 6. Evaluar con Silhouette

Calcularemos Silhouette si hay más de un cluster.

In [ ]:
# Verificamos número de clusters.
if len(np.unique(labels)) > 1:
    # Calculamos Silhouette.
    score = silhouette_score(X, labels)
    print("Silhouette Score:", score)
else:
    # Mensaje si no se puede calcular.
    print("No se puede calcular Silhouette con un solo cluster")


### Interpretación

Silhouette complementa la visualización y permite evaluar cohesión y separación.

## Bloque 7. Comparar distintos bandwidths

Observaremos cómo cambia el número de clusters al modificar bandwidth.

In [ ]:
# Definimos multiplicadores.
multipliers = [0.5, 1.0, 1.5, 2.0]

# Creamos subgráficas.
fig, axes = plt.subplots(1, len(multipliers), figsize=(18,4))

# Recorremos multiplicadores.
for ax, m in zip(axes, multipliers):
    # Creamos modelo con bandwidth modificado.
    ms = MeanShift(bandwidth=bandwidth*m, bin_seeding=True)
    
    # Entrenamos y predecimos.
    pred = ms.fit_predict(X)
    
    # Graficamos resultado.
    ax.scatter(X[:,0], X[:,1], c=pred, cmap="viridis", edgecolors="k")
    
    # Agregamos título con número de clusters.
    ax.set_title(f"bw x {m} | clusters: {len(np.unique(pred))}")

# Ajustamos figura.
plt.tight_layout()

# Mostramos figura.
plt.show()


### Interpretación

La escala de densidad define cuántos grupos aparecen. Esta es la decisión central en Mean Shift.

## Cierre

Mean Shift descubre clusters por densidad y evita definir K, pero puede ser costoso y sensible al bandwidth.